# Rossmann Retail Sales Forecasting & FP&A Analytics
## 03 — Feature Engineering and Temporal Validation

### Phase objective

This notebook converts the audited raw data into a modelling-ready feature table and defines a chronological validation strategy that resembles the Kaggle forecast horizon.

### First-block scope

- Load the raw train, test, and store files
- Build the same deterministic features for train and test
- Exclude target leakage such as `Customers`
- Validate feature completeness and train-test consistency
- Define a 48-day chronological validation period
- Restrict the principal validation population to stores included in the test set
- Save a feature dictionary and split summary

> This first block does not train a model. Stop after the validation checks and review the outputs before creating the baseline models.

## 1. Imports and project paths

Reusable transformations are stored in `src/features.py`. Keeping feature logic outside the notebook ensures that later modelling notebooks can apply exactly the same transformations.

In [1]:
from pathlib import Path
import platform
import sys

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

Python: 3.11.15
pandas: 2.3.3
NumPy: 2.4.6


In [2]:
def find_project_root(start_path: Path) -> Path:
    start_path = start_path.resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Confirm that data/ and notebooks/ exist."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORT_TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
SRC_DIR = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Source folder: {SRC_DIR}")

Project root: C:\Users\koldo\Desktop\Máster de DATA Science con IA\PROYECTO\rossmann-sales-forecasting
Source folder: C:\Users\koldo\Desktop\Máster de DATA Science con IA\PROYECTO\rossmann-sales-forecasting\src


In [3]:
from src.features import (
    MODEL_CATEGORICAL_FEATURES,
    MODEL_FEATURES,
    MODEL_NUMERIC_FEATURES,
    build_feature_table,
)

print(f"Numeric model features: {len(MODEL_NUMERIC_FEATURES)}")
print(f"Categorical model features: {len(MODEL_CATEGORICAL_FEATURES)}")
print(f"Total model features: {len(MODEL_FEATURES)}")

Numeric model features: 29
Categorical model features: 5
Total model features: 34


## 2. Load the raw datasets

The feature pipeline starts from the original files. It does not depend on manually modified intermediate datasets.

In [4]:
train = pd.read_csv(RAW_DATA_DIR / "train.csv", low_memory=False)
test = pd.read_csv(RAW_DATA_DIR / "test.csv", low_memory=False)
store = pd.read_csv(RAW_DATA_DIR / "store.csv", low_memory=False)

raw_overview = pd.DataFrame(
    {
        "dataset": ["train", "test", "store"],
        "rows": [len(train), len(test), len(store)],
        "columns": [train.shape[1], test.shape[1], store.shape[1]],
    }
)

display(raw_overview)

,dataset,rows,columns
0,train,1017209,9
1,test,41088,8
2,store,1115,10


## 3. Build the feature tables

The same function is applied to train and test. It creates only information available at forecast time and deliberately excludes predictors derived from `Sales` or `Customers`.

In [5]:
train_features = build_feature_table(train, store, dataset_name="train")
test_features = build_feature_table(test, store, dataset_name="test")

feature_table_overview = pd.DataFrame(
    {
        "dataset": ["train_features", "test_features"],
        "rows": [len(train_features), len(test_features)],
        "columns": [train_features.shape[1], test_features.shape[1]],
        "minimum_date": [train_features["Date"].min(), test_features["Date"].min()],
        "maximum_date": [train_features["Date"].max(), test_features["Date"].max()],
        "stores": [train_features["Store"].nunique(), test_features["Store"].nunique()],
    }
)

display(feature_table_overview)

,dataset,rows,columns,minimum_date,maximum_date,stores
0,train_features,1017209,47,2013-01-01,2015-07-31,1115
1,test_features,41088,46,2015-08-01,2015-09-17,856


## 4. Validate the missing `Open` treatment

The original field is retained, while `OpenMissingFlag` records the imputation and `OpenFilled` contains the modelling value.

In [6]:
open_validation = pd.DataFrame(
    {
        "dataset": ["train", "test"],
        "original_missing_open": [
            train_features["Open"].isna().sum(),
            test_features["Open"].isna().sum(),
        ],
        "imputed_open_missing": [
            train_features["OpenFilled"].isna().sum(),
            test_features["OpenFilled"].isna().sum(),
        ],
        "imputation_flags": [
            train_features["OpenMissingFlag"].sum(),
            test_features["OpenMissingFlag"].sum(),
        ],
    }
)

display(open_validation)

display(
    test_features.loc[
        test_features["OpenMissingFlag"] == 1,
        ["Id", "Store", "Date", "DayOfWeek", "Open", "OpenFilled",
         "OpenMissingFlag", "Promo", "StateHoliday", "SchoolHoliday"],
    ]
)

assert test_features["OpenFilled"].isna().sum() == 0
assert (
    test_features.loc[test_features["OpenMissingFlag"] == 1, "OpenFilled"] == 1
).all()

,dataset,original_missing_open,imputed_open_missing,imputation_flags
0,train,0,0,0
1,test,11,0,11


,Id,Store,Date,DayOfWeek,Open,OpenFilled,OpenMissingFlag,Promo,StateHoliday,SchoolHoliday
479,480,622,2015-09-17,4,NaN,1,1,1,0,0
1335,1336,622,2015-09-16,3,NaN,1,1,1,0,0
2191,2192,622,2015-09-15,2,NaN,1,1,1,0,0
3047,3048,622,2015-09-14,1,NaN,1,1,1,0,0
4759,4760,622,2015-09-12,6,NaN,1,1,0,0,0
5615,5616,622,2015-09-11,5,NaN,1,1,0,0,0
6471,6472,622,2015-09-10,4,NaN,1,1,0,0,0
7327,7328,622,2015-09-09,3,NaN,1,1,0,0,0
8183,8184,622,2015-09-08,2,NaN,1,1,0,0,0
9039,9040,622,2015-09-07,1,NaN,1,1,0,0,0


## 5. Validate feature completeness and consistency

The model feature list must exist in both datasets, contain no missing values, exclude leakage, and use the same dtypes.

In [7]:
missing_feature_columns = [
    column
    for column in MODEL_FEATURES
    if column not in train_features.columns or column not in test_features.columns
]

train_feature_missing = (
    train_features[MODEL_FEATURES].isna().sum().rename("train_missing")
)
test_feature_missing = (
    test_features[MODEL_FEATURES].isna().sum().rename("test_missing")
)

feature_missing_summary = (
    pd.concat([train_feature_missing, test_feature_missing], axis=1)
    .reset_index(names="feature")
)

feature_dtype_summary = pd.DataFrame(
    {
        "feature": MODEL_FEATURES,
        "train_dtype": [str(train_features[column].dtype) for column in MODEL_FEATURES],
        "test_dtype": [str(test_features[column].dtype) for column in MODEL_FEATURES],
    }
)
feature_dtype_summary["same_dtype"] = (
    feature_dtype_summary["train_dtype"] == feature_dtype_summary["test_dtype"]
)

print(f"Missing required feature columns: {missing_feature_columns}")
print(
    "Total missing model-feature values in train: "
    f"{feature_missing_summary['train_missing'].sum():,}"
)
print(
    "Total missing model-feature values in test: "
    f"{feature_missing_summary['test_missing'].sum():,}"
)

display(feature_missing_summary)
display(feature_dtype_summary)

assert missing_feature_columns == []
assert feature_missing_summary["train_missing"].sum() == 0
assert feature_missing_summary["test_missing"].sum() == 0
assert feature_dtype_summary["same_dtype"].all()

Missing required feature columns: []
Total missing model-feature values in train: 0
Total missing model-feature values in test: 0


,feature,train_missing,test_missing
0,DayOfWeek,0,0
1,Promo,0,0
2,SchoolHoliday,0,0
3,Year,0,0
4,Quarter,0,0
5,Month,0,0
6,Day,0,0
7,ISOWeek,0,0
8,IsWeekend,0,0
9,IsMonthStart,0,0


,feature,train_dtype,test_dtype,same_dtype
0,DayOfWeek,int64,int64,True
1,Promo,int64,int64,True
2,SchoolHoliday,int64,int64,True
3,Year,int16,int16,True
4,Quarter,int8,int8,True
5,Month,int8,int8,True
6,Day,int8,int8,True
7,ISOWeek,int16,int16,True
8,IsWeekend,int8,int8,True
9,IsMonthStart,int8,int8,True


## 6. Leakage controls

`Customers` is strongly related to sales but is unavailable in the test data and would create target leakage.

In [8]:
forbidden_predictors = {"Sales", "Customers", "OpenZeroSalesFlag", "Date", "Id"}

leakage_check = pd.DataFrame(
    {
        "forbidden_feature": sorted(forbidden_predictors),
        "included_in_model_features": [
            feature in MODEL_FEATURES for feature in sorted(forbidden_predictors)
        ],
    }
)

display(leakage_check)
assert not leakage_check["included_in_model_features"].any()

,forbidden_feature,included_in_model_features
0,Customers,False
1,Date,False
2,Id,False
3,OpenZeroSalesFlag,False
4,Sales,False


## 7. Feature dictionary

The feature dictionary documents which variables are numeric or categorical and their role in the model.

In [9]:
feature_role_notes = {
    "Store": "Store identifier; captures structural differences between stores.",
    "Promo": "Standard promotion known for the forecast date.",
    "SchoolHoliday": "School-holiday indicator known for the forecast date.",
    "CompetitionDistanceImputed": "Nearest-competitor distance with median imputation.",
    "LogCompetitionDistance": "Log transformation of competitor distance.",
    "CompetitionActive": "Whether the competitor had opened by the observation date.",
    "CompetitionMonthsOpen": "Months since competitor opening, zero when inactive or unknown.",
    "Promo2Active": "Whether Promo2 is active in the store's scheduled month.",
    "Promo2MonthsSinceStart": "Months since Promo2 started, zero before activation.",
    "StateHoliday": "State-holiday category.",
    "StoreType": "Coded Rossmann store format.",
    "Assortment": "Coded assortment type.",
    "PromoInterval": "Recurring Promo2 month schedule.",
}

feature_dictionary = pd.DataFrame(
    {
        "feature": MODEL_FEATURES,
        "feature_type": [
            "numeric" if feature in MODEL_NUMERIC_FEATURES else "categorical"
            for feature in MODEL_FEATURES
        ],
        "dtype": [str(train_features[feature].dtype) for feature in MODEL_FEATURES],
        "description": [
            feature_role_notes.get(
                feature,
                "Deterministic calendar or store-level feature available at forecast time.",
            )
            for feature in MODEL_FEATURES
        ],
    }
)

display(feature_dictionary)

,feature,feature_type,dtype,description
0,DayOfWeek,numeric,int64,Deterministic calendar or store-level feature ...
1,Promo,numeric,int64,Standard promotion known for the forecast date.
2,SchoolHoliday,numeric,int64,School-holiday indicator known for the forecas...
3,Year,numeric,int16,Deterministic calendar or store-level feature ...
4,Quarter,numeric,int8,Deterministic calendar or store-level feature ...
5,Month,numeric,int8,Deterministic calendar or store-level feature ...
6,Day,numeric,int8,Deterministic calendar or store-level feature ...
7,ISOWeek,numeric,int16,Deterministic calendar or store-level feature ...
8,IsWeekend,numeric,int8,Deterministic calendar or store-level feature ...
9,IsMonthStart,numeric,int8,Deterministic calendar or store-level feature ...


## 8. Define the chronological validation horizon

The Kaggle test covers 48 consecutive dates. The validation period therefore uses the final 48 dates of the historical data: 14 June 2015 to 31 July 2015.

The principal evaluation population is restricted to stores that also appear in the test set, while training may still use all stores available before the cut-off.

In [10]:
test_horizon_days = test_features["Date"].nunique()
validation_end = train_features["Date"].max()
validation_start = validation_end - pd.Timedelta(days=test_horizon_days - 1)

train_period = train_features.loc[
    train_features["Date"] < validation_start
].copy()

validation_period = train_features.loc[
    train_features["Date"].between(validation_start, validation_end)
].copy()

test_store_ids = set(test_features["Store"].unique())
validation_test_stores = validation_period.loc[
    validation_period["Store"].isin(test_store_ids)
].copy()

split_summary = pd.DataFrame(
    {
        "split": ["training", "validation_all_stores", "validation_test_stores", "kaggle_test"],
        "minimum_date": [
            train_period["Date"].min(), validation_period["Date"].min(),
            validation_test_stores["Date"].min(), test_features["Date"].min(),
        ],
        "maximum_date": [
            train_period["Date"].max(), validation_period["Date"].max(),
            validation_test_stores["Date"].max(), test_features["Date"].max(),
        ],
        "rows": [
            len(train_period), len(validation_period),
            len(validation_test_stores), len(test_features),
        ],
        "unique_dates": [
            train_period["Date"].nunique(), validation_period["Date"].nunique(),
            validation_test_stores["Date"].nunique(), test_features["Date"].nunique(),
        ],
        "stores": [
            train_period["Store"].nunique(), validation_period["Store"].nunique(),
            validation_test_stores["Store"].nunique(), test_features["Store"].nunique(),
        ],
    }
)

print(f"Test horizon: {test_horizon_days} days")
print(f"Validation start: {validation_start.date()}")
print(f"Validation end: {validation_end.date()}")
display(split_summary)

Test horizon: 48 days
Validation start: 2015-06-14
Validation end: 2015-07-31


,split,minimum_date,maximum_date,rows,unique_dates,stores
0,training,2013-01-01,2015-06-13,963689,894,1115
1,validation_all_stores,2015-06-14,2015-07-31,53520,48,1115
2,validation_test_stores,2015-06-14,2015-07-31,41088,48,856
3,kaggle_test,2015-08-01,2015-09-17,41088,48,856


## 9. Define modelling and evaluation rows

Closed stores will receive a final prediction of zero, so the sales model will be trained on open-store observations. Rows with actual sales equal to zero are excluded only from percentage-based metrics.

In [11]:
training_open_rows = train_period.loc[
    train_period["OpenFilled"] == 1
].copy()

validation_open_rows = validation_test_stores.loc[
    validation_test_stores["OpenFilled"] == 1
].copy()

validation_metric_rows = validation_test_stores.loc[
    (validation_test_stores["OpenFilled"] == 1)
    & (validation_test_stores["Sales"] > 0)
].copy()

modelling_population_summary = pd.DataFrame(
    {
        "population": [
            "training_open_rows", "validation_test_store_rows",
            "validation_open_rows", "validation_metric_rows",
            "validation_closed_rows", "validation_open_zero_sales_rows",
        ],
        "rows": [
            len(training_open_rows), len(validation_test_stores),
            len(validation_open_rows), len(validation_metric_rows),
            int((validation_test_stores["OpenFilled"] == 0).sum()),
            int(
                (
                    (validation_test_stores["OpenFilled"] == 1)
                    & (validation_test_stores["Sales"] == 0)
                ).sum()
            ),
        ],
    }
)

display(modelling_population_summary)

,population,rows
0,training_open_rows,798508
1,validation_test_store_rows,41088
2,validation_open_rows,35262
3,validation_metric_rows,35262
4,validation_closed_rows,5826
5,validation_open_zero_sales_rows,0


## 10. Temporal split validation

These controls ensure that no future date leaks into training, the horizon contains 48 dates, every validation store has prior history, and all model features are complete.

In [12]:
validation_store_ids = set(validation_test_stores["Store"].unique())
training_store_ids = set(train_period["Store"].unique())

split_controls = pd.DataFrame(
    {
        "control": [
            "Training ends before validation starts",
            "Validation contains 48 unique dates",
            "All validation stores have training history",
            "No model-feature missing values in training open rows",
            "No model-feature missing values in validation metric rows",
            "No model-feature missing values in test",
        ],
        "passed": [
            train_period["Date"].max() < validation_start,
            validation_period["Date"].nunique() == test_horizon_days,
            validation_store_ids.issubset(training_store_ids),
            not training_open_rows[MODEL_FEATURES].isna().any().any(),
            not validation_metric_rows[MODEL_FEATURES].isna().any().any(),
            not test_features[MODEL_FEATURES].isna().any().any(),
        ],
        "observed_value": [
            f"{train_period['Date'].max().date()} < {validation_start.date()}",
            validation_period["Date"].nunique(),
            len(validation_store_ids - training_store_ids),
            int(training_open_rows[MODEL_FEATURES].isna().sum().sum()),
            int(validation_metric_rows[MODEL_FEATURES].isna().sum().sum()),
            int(test_features[MODEL_FEATURES].isna().sum().sum()),
        ],
    }
)

display(split_controls)
assert split_controls["passed"].all()

,control,passed,observed_value
0,Training ends before validation starts,True,2015-06-13 < 2015-06-14
1,Validation contains 48 unique dates,True,48
2,All validation stores have training history,True,0
3,No model-feature missing values in training op...,True,0
4,No model-feature missing values in validation ...,True,0
5,No model-feature missing values in test,True,0


## 11. Save the phase outputs

Only compact documentation tables are saved to GitHub. The full feature tables remain reproducible from the raw files.

In [14]:
REPORT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

feature_dictionary_path = REPORT_TABLES_DIR / "feature_dictionary_phase3.csv"
split_summary_path = REPORT_TABLES_DIR / "temporal_validation_split_summary.csv"
modelling_population_path = REPORT_TABLES_DIR / "modelling_population_summary.csv"
split_controls_path = REPORT_TABLES_DIR / "temporal_validation_controls.csv"

feature_dictionary.to_csv(feature_dictionary_path, index=False)
split_summary.to_csv(split_summary_path, index=False)
modelling_population_summary.to_csv(modelling_population_path, index=False)
split_controls.to_csv(split_controls_path, index=False)

print("Saved:")
print(f"- {feature_dictionary_path.relative_to(PROJECT_ROOT)}")
print(f"- {split_summary_path.relative_to(PROJECT_ROOT)}")
print(f"- {modelling_population_path.relative_to(PROJECT_ROOT)}")
print(f"- {split_controls_path.relative_to(PROJECT_ROOT)}")

Saved:
- reports\tables\feature_dictionary_phase3.csv
- reports\tables\temporal_validation_split_summary.csv
- reports\tables\modelling_population_summary.csv
- reports\tables\temporal_validation_controls.csv


## Stop and review

Before creating baseline models, share:

1. `feature_table_overview`
2. `open_validation`
3. The totals from `feature_missing_summary`
4. `feature_dtype_summary` if any `same_dtype` value is false
5. `split_summary`
6. `modelling_population_summary`
7. `split_controls`
8. Any warning or error

### Questions to answer together

- Were the same deterministic features created for train and test?
- Are all model features complete and type-consistent?
- Does the 48-day validation period correctly imitate the Kaggle horizon?
- Should the principal validation score use only the 856 test stores?
- Should the main model train on all open observations or exclude the 54 open-zero-sales rows?
- Is the feature table ready for the baseline models?